In [1]:
from alb_sim.config.run import RunConfig
from alb_sim.config.sea_floor import SeaFloorConfig
from alb_sim.config.sensor import SensorConfig
from alb_sim.config.simulation import SimulationConfig
from alb_sim.config.water import TurbidityLayerConfig, WaterConfig
from alb_sim.execution.parallel import merge_results, run_parallel
from alb_sim.utils.parameter_profile import ExponentialParameter

Define a multi-layer water column, sea surface, and seafloor configuration, along with parallel run settings (batches and processes).

In [ ]:
simulation_config = SimulationConfig(
    water=WaterConfig(
        layers=(
            TurbidityLayerConfig(
                height=0.4, absorption_coefficient=0.9, scattering_coefficient=1.21
            ),
            TurbidityLayerConfig(
                height=2, absorption_coefficient=0.2, scattering_coefficient=1.21
            ),
            TurbidityLayerConfig(
                height=0.3,
                absorption_coefficient=ExponentialParameter(start=0.2, end=0.5),
                scattering_coefficient=ExponentialParameter(start=1.21, end=2),
                sublayer_dz=None,
                num_sublayers=256,
            ),
            TurbidityLayerConfig(
                height=4.3,
                absorption_coefficient=0.5,
                scattering_coefficient=2,
            ),
        )
    ),
    sensor=SensorConfig(field_of_view=18e-3, aperture_radius=0.1),
    sea_floor=SeaFloorConfig(albedo=0.1),
)
run_config = RunConfig(
    processes=8,
    batches_forward=(8) * 3,
    batches_backward=(8) * 5,
)

Either: Run a single parallel simulation to produce a waveform and return any additional outputs.

In [ ]:
waveform, _, _ = run_parallel(simulation_config, run_config)

Starting forwards pass


Or: Run the parallel simulation multiple times, then merge individual waveforms into an aggregate result for improved statistics.

In [ ]:
waveforms, _, _ = [run_parallel(simulation_config, run_config) for _ in range(2)]

waveform = merge_results(waveforms)

Visualise the stacked waveform contributions per photon type, optionally overlaying water-layer boundaries for context.

In [ ]:
from alb_sim.plotting.plot_stacked_waveform import plot_stacked_waveform
from alb_sim.utils.water_layer_steps import get_water_layer_steps

show_layer_lines = False
x_limits = None  # (low, high)
plot_stacked_waveform(
    waveform,
    xlim=x_limits,
    layer_lines=get_water_layer_steps(simulation_config) if show_layer_lines else None,
)

Plot the total waveform summed over photon types to inspect the overall depth response, again with optional layer markers.

In [ ]:
import numpy as np

from alb_sim.plotting.plot_waveform import plot_waveform
from alb_sim.utils.water_layer_steps import get_water_layer_steps

show_layer_lines = False
x_limits = None  # (low, high)
plot_waveform(
    np.sum(list(waveform.values()), axis=0),
    xlim=x_limits,
    layer_lines=get_water_layer_steps(simulation_config) if show_layer_lines else None,
)

Persist the final merged waveform to disk as a pickle file for later analysis or plotting outside this notebook.

In [ ]:
import pickle

with open(r"waveform.pickle", "wb") as output_file:
    pickle.dump(waveform, output_file)